# Q&A notebook

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

## How many virus detections were reported in McLeish2024?

In [ ]:

len(db.conn.sql('SELECT * FROM D_virusHits').df()) # 1616 hits in our database q000016

## How many taxa?

In [ ]:
len(db.conn.sql('SELECT * FROM D_virusHits').df()['scientific_name'].unique()) # 158 hits in our database q000017

## Did all the PAB bacteria detected correspond to at least a virus in their library?

In [ ]:
virus = db.conn.sql('SELECT * FROM D_virusHits').df()
bacteria = db.conn.sql('SELECT * FROM D_pabHits').df()
virus_positive_libraries = virus['library'].unique()
bacteria['is_in_virus'] = bacteria['library'].isin(virus_positive_libraries)
bacteria.groupby(['scientific_name'], as_index=False)['is_in_virus'].sum().query('is_in_virus != 0')

In [ ]:
len(bacteria.taxid.unique())

## Were all the virus detected in the pressence of a bacteria?

In [ ]:
virus = db.conn.sql('SELECT * FROM D_virusHits').df()
bacteria = db.conn.sql('SELECT * FROM D_pabHits').df()
bacteria_positive_libraries = bacteria['library'].unique()
virus['is_in_bacteria'] = virus['library'].isin(bacteria_positive_libraries)
virus.groupby(['scientific_name'], as_index=False)['is_in_bacteria'].sum().query('is_in_bacteria != 0')

## How many libraries has the site with most libraries?

In [ ]:
pd.read_csv('output/metadata.site-library.csv', sep=';').value_counts('site')

## What's the virus with wider range?

In [ ]:
m = pd.merge(
    pd.read_csv("output/hits.virus.csv", sep=';'),
    pd.read_csv('output/metadata.site-library.csv', sep=';'),
    on='library'
)[['acronym', 'taxid', 'host_taxon']].drop_duplicates().value_counts('acronym').reset_index()

In [ ]:
m

In [ ]:
m.query('count == 1')

In [ ]:
m = pd.merge(
    pd.read_csv("output/hits.bacteria.csv", sep=';').query('is_pab == True'),
    pd.read_csv('output/metadata.site-library.csv', sep=';'),
    on='library'
)[['scientific_name', 'taxid', 'host_taxon']].drop_duplicates().value_counts('scientific_name').reset_index()

In [ ]:
len(m.query('count == 1'))

In [ ]:
db.conn.close()